In [ ]:
from sympy import symbols
from sympy.logic.boolalg import And, Or, Not, Implies, Equivalent, simplify_logic, to_cnf, to_dnf, to_nnf
from sympy.parsing.sympy_parser import parse_expr

_ATOMS = symbols('p q r s t u v w x y z m', boolean=True)
_NAME2SYM = {str(s): s for s in _ATOMS}

def _parse_node(s: str):
    s = s.strip().replace('↔', '<->')

    def _to_implies(text):
        i = text.find('->')
        if i == -1: return text
        def left_bound(j):
            if text[j-1] != ')':
                k = j-1
                while k>0 and (text[k-1].isalnum() or text[k-1] in ['~']):
                    k -= 1
                return k, j-1
            bal = 1; k = j-2
            while k>=0 and bal>0:
                if text[k] == ')': bal += 1
                elif text[k] == '(': bal -= 1
                k -= 1
            return k+1, j-1
        def right_bound(j):
            if j+2 < len(text) and text[j+2] != '(':
                k = j+2
                while k < len(text) and (text[k].isalnum() or text[k] in ['~']):
                    k += 1
                return j+2, k-1
            bal = 1; k = j+3
            while k < len(text) and bal>0:
                if text[k] == '(': bal += 1
                elif text[k] == ')': bal -= 1
                k += 1
            return j+2, k-1
        L0, L1 = left_bound(i); R0, R1 = right_bound(i)
        left = text[L0:L1+1]; right = text[R0:R1+1]
        new = text[:L0] + f'Implies({left},{right})' + text[R1+1:]
        return _to_implies(new)

    def _to_equiv(text):
        j = text.find('<->')
        if j == -1: return text
        left, right = text[:j].rstrip(), text[j+3:].lstrip()
        return f'Equivalent({left},{right})'
    return parse_expr(_to_equiv(_to_implies(s)), local_dict=_NAME2SYM, evaluate=False)

def _stringify(e): return str(e)

def rewrite_node(node_str: str, *,
                 budget: int = 16,
                 do_simplify: bool = True,
                 do_nnf: bool = True,
                 do_imp_elim: bool = True,
                 do_imp_intro: bool = True,
                 do_eqv_elim: bool = True,
                 do_eqv_intro: bool = True,
                 do_cnf: bool = True,
                 do_dnf: bool = False) -> list[str]:

    expr = _parse_node(node_str)
    seen, out = set(), []

    def _add(e):
        try:
            s = _stringify(e)
        except Exception:
            return
        if s not in seen:
            seen.add(s)
            out.append(s)

    _add(expr)

    # A) simlify
    if do_simplify:
        try: _add(simplify_logic(expr, force=True))
        except: pass

    # B) NNF
    if do_nnf:
        try: _add(to_nnf(expr, simplify=True))
        except: pass

    # C) → ：Imp(a,b) -> (¬a ∨ b)
    if do_imp_elim:
        cand = expr.replace(lambda x: isinstance(x, Implies),
                            lambda x: Or(Not(x.args[0]), x.args[1]))
        _add(cand)

    # D) ↔ 
    if do_eqv_elim:
        cand1 = expr.replace(lambda x: isinstance(x, Equivalent),
                             lambda x: And(Implies(x.args[0], x.args[1]),
                                           Implies(x.args[1], x.args[0])))
        _add(cand1)
        cand2 = expr.replace(lambda x: isinstance(x, Equivalent),
                             lambda x: Or(And(x.args[0], x.args[1]),
                                          And(Not(x.args[0]), Not(x.args[1]))))
        _add(cand2)

    # E) → ：Or(Not(a), b) -> Implies(a,b)
    if do_imp_intro:
        def intro_impl(e):
            if isinstance(e, Or) and len(e.args) == 2:
                a, b = e.args
                if isinstance(a, Not): return Implies(a.args[0], b)
                if isinstance(b, Not): return Implies(b.args[0], a)
            return e
        cand = expr.replace(lambda x: isinstance(x, Or), intro_impl)
        _add(cand)

    # F) ↔ ：(a&b) | (~a&~b) -> Equivalent(a,b)
    if do_eqv_intro:
        def intro_equiv(e):
            if isinstance(e, Or) and len(e.args) == 2:
                A, B = e.args
                if isinstance(A, And) and isinstance(B, And) and len(A.args) == 2 and len(B.args) == 2:
                    a1, a2 = A.args; b1, b2 = B.args
                    for a, b in ((a1,a2),(a2,a1)):
                        if isinstance(b1, Not) and isinstance(b2, Not) and {b1.args[0], b2.args[0]} == {a, b}:
                            return Equivalent(a, b)
            return e
        cand = expr.replace(lambda x: isinstance(x, Or), intro_equiv)
        _add(cand)

    # G) CNF / DNF
    if do_cnf:
        try: _add(to_cnf(expr, simplify=True, force=True))
        except: pass
    if do_dnf:
        try: _add(to_dnf(expr, simplify=True, force=True))
        except: pass

    return out[:max(1, budget)]

In [ ]:
nodes = [
    "r | ~p"
]
for n in nodes:
    print("INPUT:", n)
    res = rewrite_node(n, budget=12, do_dnf=True)  
    for i, r in enumerate(res):
        print(f"  [{i}] {r}")

INPUT: r | ~p
  [0] r | ~p
  [1] Implies(p, r)
